## Day 13 test plan (written before opening any test file)

### 1. Models (frozen files)
- **LR: full-text B-none** — `tfidf_b.joblib` + `logreg_b.joblib` (val 0.696). This counts in the rule because it is the LR we would actually deploy, it is the stricter bar, and the 0.6 review threshold was chosen on it. Truncated B-350 (val 0.694) is reported as a fairness reference only.
- **LSTM:** `lstm_best.keras` + `lstm_vocab.json` (val 0.6735)
- **BERT:** `bert_best.keras`, epoch 3 (val 0.7205)
- All use `label_order.json`. Pipelines are frozen: `clean_text` → (`first_350` for LSTM/BERT/B-350) → (`format_for_bert` for BERT only).

### 2. Scores
- **Score 1: full test** (169,189 rows): what operations will see, including repeated template complaints.
- **Score 2: test_dedup** (87,091 rows): does the model understand complaints it has not effectively seen before?
- **The rule is applied to test_dedup first**, because it measures generalisation rather than memorised templates. It must **also pass on the full test**. **If the two disagree, LR stays**, and I report both.

### 3. Metrics
- Primary: **macro-F1**. Also: accuracy and per-class F1 for all three models, on both scores.

### 4. Decision rule (as written on Day 12)
BERT is deployed only if it:
1. beats LR by **> 0.01 macro-F1**,
2. has **no small class more than 0.02 below LR**. The small classes are fixed now, from val: Debt or credit management, Prepaid card, Payday loan, Vehicle loan, Student loan, Mortgage,
3. **fits in 512 MB** via ONNX,
4. answers in **< 200 ms per ticket** on Render.
Otherwise LR stays.

### 5. Commitments
- The test set is opened **once**. All 3 models are scored in one run, and predictions are saved to disk.
- **No retraining, tuning, threshold or pipeline changes** after seeing test results.
- The **0.6 review threshold stays** as chosen on validation. I only report how it performs on test.
- If BERT fails the rule on test, I report it honestly and **LR stays**, even though validation said otherwise.


In [3]:
import hashlib, json, datetime, sklearn, pandas as pd, tensorflow as tf, keras, keras_hub

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

local_files = ["models/tfidf_b.joblib", "models/logreg_b.joblib",
               "models/tfidf_b350.joblib", "models/logreg_b350.joblib",
               "models/lstm_best.keras", "models/lstm_vocab.json",
               "models/label_order.json", "src/text.py"]

manifest = {
    "date": str(datetime.date.today()),
    "checksums": {f: sha256(f"../{f}") for f in local_files},
    "versions": {
        "local (LR, LSTM)": {"sklearn": sklearn.__version__, "tensorflow": tf.__version__,
                             "keras": keras.__version__, "keras_hub": keras_hub.__version__,
                             "pandas": pd.__version__},
        "colab (BERT)": {"tensorflow": "2.21.0", "keras": "3.13.2", "keras_hub": "0.26.0", "pandas": "2.2.3"},
                  # paste the dict printed by Cell A
    },
}
manifest["checksums"]["models/bert_best.keras"]= "58dab6bbc8b8dcbd469243825eb59bf77cbe1882ac7c7d941d02e8a91d4bd66a"

with open("../models/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# determinism check ✅
assert sha256("../models/lstm_best.keras") == manifest["checksums"]["models/lstm_best.keras"]
for f, h in manifest["checksums"].items():
    print(f"{f:32} {h[:12]}")


models/tfidf_b.joblib            29a86e481ab5
models/logreg_b.joblib           240d83fddbbe
models/tfidf_b350.joblib         8fd0de4babfb
models/logreg_b350.joblib        a85d5c0e4157
models/lstm_best.keras           f39fb0b6f7b3
models/lstm_vocab.json           7d96f49aa8d7
models/label_order.json          8e0f2f852f3d
src/text.py                      26f00e8b982d
models/bert_best.keras           58dab6bbc8b8


In [4]:
import sys; sys.path.append("..")
import json, joblib, numpy as np, pandas as pd, tensorflow as tf, keras
from src.text import clean_text

def first_350(text):
    return " ".join(text.split()[:350])

label_order = json.load(open("../models/label_order.json"))
label_to_id = {l: i for i, l in enumerate(label_order)}

vec_full = joblib.load("../models/tfidf_b.joblib")
vec_350  = joblib.load("../models/tfidf_b350.joblib")
vocab    = json.load(open("../models/lstm_vocab.json"))
vec_lstm = keras.layers.TextVectorization(max_tokens=25_000, output_sequence_length=350, vocabulary=vocab)

size_full, size_350 = len(vec_full.vocabulary_), len(vec_350.vocabulary_)   # BEFORE

X_full, X_350, lstm_ds, y_test = {}, {}, {}, {}
for name in ["test", "test_dedup"]:
    df = pd.read_parquet(f"../data/processed/{name}.parquet")
    y = df["label"].map(label_to_id)
    print(name, df.shape, "| NaN labels:", y.isna().sum())
    y_test[name] = y.values


test (169189, 5) | NaN labels: 0
test_dedup (87091, 5) | NaN labels: 0


In [5]:
X_full, X_350, lstm_ds = {}, {}, {}
for name in ["test", "test_dedup"]:
    df = pd.read_parquet(f"../data/processed/{name}.parquet")
    clean = df["Consumer complaint narrative"].map(clean_text)
    X_full[name] = vec_full.transform(clean)                         # .transform ONLY
    X_350[name]  = vec_350.transform(clean.map(first_350))
    lstm_ds[name] = (tf.data.Dataset.from_tensor_slices(clean.values)
                     .batch(256).map(vec_lstm).prefetch(tf.data.AUTOTUNE))   # no shuffle

assert len(vec_full.vocabulary_) == size_full == 24_739, "Full TF-IDF changed!"
assert len(vec_350.vocabulary_) == size_350, "B-350 TF-IDF changed!"
assert len(vec_lstm.get_vocabulary()) == 25_000, "LSTM vocab changed!"
print(f"Vocab asserts passed ✅ | full {size_full} | b350 {size_350} | lstm 25000")


Vocab asserts passed ✅ | full 24739 | b350 22023 | lstm 25000


In [6]:
import os
os.makedirs("../predictions", exist_ok=True)

def check_and_save(model_name, set_name, proba):
    proba = np.asarray(proba, dtype="float32")
    pred = proba.argmax(axis=1)
    y = y_test[set_name]
    assert len(pred) == len(y), "Row count mismatch!"
    assert np.allclose(proba.sum(axis=1), 1, atol=1e-3), "Rows don't sum to 1!"
    assert (proba.argmax(axis=1) == pred).all()
    np.save(f"../predictions/proba_{model_name}_{set_name}.npy", proba)
    np.save(f"../predictions/pred_{model_name}_{set_name}.npy", pred)
    print(f"✅ {model_name:7} {set_name:10} proba {proba.shape} pred {pred.shape}")

lr_full = joblib.load("../models/logreg_b.joblib")
lr_350  = joblib.load("../models/logreg_b350.joblib")
lstm    = keras.models.load_model("../models/lstm_best.keras")

assert list(lr_full.classes_) == label_order, "LR column order ≠ label_order!"
assert list(lr_350.classes_) == label_order, "LR-350 column order ≠ label_order!"

for s in ["test", "test_dedup"]:
    np.save(f"../predictions/y_{s}.npy", y_test[s])
    check_and_save("lr_full", s, lr_full.predict_proba(X_full[s]))
    check_and_save("lr_350",  s, lr_350.predict_proba(X_350[s]))
    check_and_save("lstm",    s, lstm.predict(lstm_ds[s], verbose=0))   # already softmax

print("Local predictions saved 🔒 (no metrics computed)")


✅ lr_full test       proba (169189, 11) pred (169189,)
✅ lr_350  test       proba (169189, 11) pred (169189,)
✅ lstm    test       proba (169189, 11) pred (169189,)
✅ lr_full test_dedup proba (87091, 11) pred (87091,)
✅ lr_350  test_dedup proba (87091, 11) pred (87091,)
✅ lstm    test_dedup proba (87091, 11) pred (87091,)
Local predictions saved 🔒 (no metrics computed)


In [7]:
from sklearn.metrics import f1_score, accuracy_score

models = {"LR full": "lr_full", "LR-350": "lr_350", "LSTM": "lstm", "BERT": "bert"}
val_f1 = {"LR full": 0.696, "LR-350": 0.694, "LSTM": 0.674, "BERT": 0.721}
sets = ["test", "test_dedup"]

y    = {s: np.load(f"../predictions/y_{s}.npy") for s in sets}
pred = {(m, s): np.load(f"../predictions/pred_{f}_{s}.npy") for m, f in models.items() for s in sets}

# Step 1: overall
rows = []
for m in models:
    r = {"Model": m, "Val macro-F1": val_f1[m]}
    for s in sets:
        r[f"{s} macro-F1"] = f1_score(y[s], pred[(m, s)], average="macro")
        r[f"{s} acc"]      = accuracy_score(y[s], pred[(m, s)])
    rows.append(r)
display(pd.DataFrame(rows).round(4))

# Step 2: per class
for s in sets:
    t = pd.DataFrame({m: f1_score(y[s], pred[(m, s)], average=None) for m in models}, index=label_order)
    t.insert(0, "support", np.bincount(y[s], minlength=11))
    t["BERT − LR full"] = t["BERT"] - t["LR full"]
    print(f"\n=== {s} ===")
    display(t.sort_values("BERT − LR full").round(3))

# Step 3: review threshold (LR full, test, 0.6 fixed on val)
proba = np.load("../predictions/proba_lr_full_test.npy")
review = proba.max(axis=1) < 0.6
wrong  = pred[("LR full", "test")] != y["test"]
print(f"\nReview queue: {review.mean():.1%} of tickets | catches {review[wrong].mean():.1%} of errors (val: 14.6% / 44.5%)")


,Model,Val macro-F1,test macro-F1,test acc,test_dedup macro-F1,test_dedup acc
0,LR full,0.696,0.6561,0.9098,0.6586,0.8604
1,LR-350,0.694,0.6564,0.9085,0.6590,0.8591
2,LSTM,0.674,0.6444,0.9046,0.6489,0.8587
3,BERT,0.721,0.6857,0.9156,0.6861,0.8665



=== test ===


,support,LR full,LR-350,LSTM,BERT,BERT − LR full
Student loan,2144,0.614,0.608,0.660,0.602,-0.013
Mortgage,1828,0.871,0.873,0.866,0.871,0.000
Credit card,5049,0.727,0.724,0.700,0.729,0.002
Credit reporting or other personal consumer reports,135456,0.958,0.957,0.955,0.961,0.003
Checking or savings account,5115,0.786,0.784,0.781,0.797,0.011
"Money transfer, virtual currency, or money service",3067,0.751,0.748,0.769,0.770,0.019
Vehicle loan or lease,1349,0.654,0.656,0.635,0.675,0.021
Prepaid card,465,0.613,0.615,0.599,0.642,0.029
Debt collection,13507,0.633,0.628,0.629,0.696,0.062
"Payday loan, title loan, personal loan, or advance loan",905,0.522,0.528,0.494,0.584,0.062



=== test_dedup ===


,support,LR full,LR-350,LSTM,BERT,BERT − LR full
Student loan,2100,0.624,0.617,0.670,0.610,-0.014
Mortgage,1828,0.873,0.875,0.868,0.874,0.000
Credit reporting or other personal consumer reports,57451,0.927,0.927,0.928,0.931,0.004
Checking or savings account,5108,0.786,0.784,0.782,0.798,0.011
Debt collection,9870,0.688,0.682,0.678,0.703,0.016
Vehicle loan or lease,1346,0.658,0.661,0.640,0.679,0.021
Credit card,4927,0.740,0.737,0.731,0.762,0.022
"Money transfer, virtual currency, or money service",2787,0.725,0.721,0.745,0.747,0.022
Prepaid card,465,0.613,0.615,0.599,0.642,0.029
"Payday loan, title loan, personal loan, or advance loan",905,0.523,0.530,0.496,0.585,0.062



Review queue: 8.8% of tickets | catches 36.6% of errors (val: 14.6% / 44.5%)


In [8]:
rng = np.random.default_rng(42)
s = "test_dedup"
y_d, p_bert, p_lr = y[s], pred[("BERT", s)], pred[("LR full", s)]
student = label_order.index("Student loan")

diffs, student_diffs = [], []
for _ in range(1000):
    idx = rng.integers(0, len(y_d), len(y_d))                  # resample WITH replacement
    f_bert = f1_score(y_d[idx], p_bert[idx], average=None, labels=range(11))
    f_lr   = f1_score(y_d[idx], p_lr[idx],   average=None, labels=range(11))
    diffs.append(f_bert.mean() - f_lr.mean())                  # macro-F1 difference
    student_diffs.append(f_bert[student] - f_lr[student])      # the close call 👀

lo, hi = np.percentile(diffs, [2.5, 97.5])
slo, shi = np.percentile(student_diffs, [2.5, 97.5])
print(f"Macro-F1 BERT − LR: 95% CI [{lo:+.4f}, {hi:+.4f}] | share > 0: {(np.array(diffs) > 0).mean():.1%}")
print(f"Student loan BERT − LR: 95% CI [{slo:+.4f}, {shi:+.4f}]")


Macro-F1 BERT − LR: 95% CI [+0.0205, +0.0344] | share > 0: 100.0%
Student loan BERT − LR: 95% CI [-0.0294, -0.0005]


In [12]:
import sys, json, hashlib, numpy as np, onnxruntime as ort
sys.path.append("..")
from tokenizers import BertWordPieceTokenizer
from scipy.special import softmax
from src.text import clean_text, format_for_bert
def first_350(t): return " ".join(t.split()[:350])

raw       = json.load(open("../models/ref_raw_texts.json"))
ref_ids   = np.load("../models/ref_token_ids.npy")
ref_probs = np.load("../models/ref_probs.npy")

# 1️⃣ tokenizer parity: the FULL local pipeline, raw text → ids (also catches pandas 2 vs 3)
tok = BertWordPieceTokenizer("../models/bert_vocab.txt", lowercase=True)
tok.enable_truncation(512)
tok.enable_padding(length=512, pad_id=0, pad_token="[PAD]")
texts = [format_for_bert(first_350(clean_text(t))) for t in raw]
ids = np.array([e.ids for e in tok.encode_batch(texts)])
same = (ids == ref_ids).all(axis=1)
print(f"Tokenizer parity: {same.mean():.1%} rows identical")
if not same.all():
    i = np.where(~same)[0][0]
    print("First mismatch row", i, "| local:", ids[i][:20], "| keras:", ref_ids[i][:20])

# 2️⃣ network parity
sess = ort.InferenceSession("../models/bert.onnx", providers=["CPUExecutionProvider"])
for inp in sess.get_inputs():
    print("ONNX input:", inp.name, inp.type, inp.shape)

np_type = {"tensor(int32)": np.int32, "tensor(int64)": np.int64, "tensor(bool)": np.bool_}
arrays  = {"token_ids": ids, "segment_ids": np.zeros_like(ids), "padding_mask": ids != 0}

def run(sl):
    feeds = {}
    for inp in sess.get_inputs():
        key = next(k for k in arrays if k in inp.name)              # matches 'token_ids' or 'token_ids:0'
        feeds[inp.name] = arrays[key][sl].astype(np_type[inp.type])
    return sess.run(None, feeds)[0]

logits = np.concatenate([run(slice(i, i + 50)) for i in range(0, len(ids), 50)])
probs_onnx = softmax(logits.astype("float32"), axis=1)
print(f"Max |prob diff|: {np.abs(probs_onnx - ref_probs).max():.5f}")
print(f"Argmax match: {(probs_onnx.argmax(1) == ref_probs.argmax(1)).mean():.1%}")

# 🔏 fingerprint the deployable files
def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

m = json.load(open("../models/manifest.json"))
for f in ["models/bert.onnx", "models/bert_vocab.txt"]:
    m["checksums"][f] = sha256(f"../{f}")
    print(f"{f:24} {m['checksums'][f][:12]}")
json.dump(m, open("../models/manifest.json", "w"), indent=2)


Tokenizer parity: 100.0% rows identical
ONNX input: padding_mask tensor(int32) ['unk__1278', 'unk__1279']
ONNX input: segment_ids tensor(int32) ['unk__1280', 'unk__1281']
ONNX input: token_ids tensor(int32) ['unk__1282', 'unk__1283']
Max |prob diff|: 0.00263
Argmax match: 100.0%
models/bert.onnx         e24c7e73d665
models/bert_vocab.txt    07eced375cec


# Decision record (Day 13, 2026-10-30)

## 1. Best model (quality)
**BERT** (`bert_small_en_uncased`, fine-tuned, 512 tokens).
- Macro-F1: **0.6857** test / **0.6861** test_dedup (LR full: 0.6561 / 0.6586).
- Paired bootstrap on test_dedup (1,000 resamples): BERT − LR = **[+0.0205, +0.0344]**, with BERT ahead in 100% of resamples.
- Why: it reads words in context instead of counting them, so it separates overlapping products better than TF-IDF's bag of words.

## 2. Deployed model (provisional)
**BERT (ONNX, real-length padding) if Render's measured p95 is under 200 ms on Nov 4; otherwise LR full-text.**
- Engineering consequence: FastAPI chooses the model from config (`params.yaml`: `model: bert` / `model: lr`). Both sets of files are packaged and their sha256 checked against `models/manifest.json` at startup, so Nov 4 is a one-line switch, not a rebuild.
- Allowed speed-ups (identical predictions): ORT graph optimisation and real-length padding (proven 100% argmax match, max |diff| 0.00263). Not allowed without full re-validation: shorter truncation, int8 quantisation, a smaller model.

## 3. Rule results
| # | Criterion | Result | Status |
|---|---|---|---|
| 1 | Macro-F1 > LR + 0.01 | +0.0275 (dedup) / +0.0296 (test) | ✅ Pass |
| 2 | No small class > 0.02 below LR | worst: Student loan −0.014 (dedup) / −0.013 (test) | ✅ Pass |
| 3 | Memory ≤ 512 MB | peak 255.7 MB locally (about 356 MB with FastAPI) | ✅ Local pass, Render pending |
| 4 | p95 < 200 ms | 179.8 ms locally (1 thread, real length); about 1.8 s projected on Render (×10) | ⏳ Pending, likely fail |

## 4. Known risks and monitoring
- **Student loan:** BERT is slightly worse than LR (bootstrap CI [−0.0294, −0.0005], entirely below 0). Monitor its F1 separately.
- **Debt or credit management:** low F1 for every model (best 0.217) because it overlaps with Debt collection.
- **Drift:** Money transfer and Student loan dropped from val to test for all models, which points to a data shift rather than a model issue.
- **Review threshold weakened:** on test, 8.8% of tickets are reviewed and 36.6% of errors caught (val: 14.6% / 44.5%). Recalibrate on fresh data.
- **Label noise ceiling:** about 8% of rows sit in conflicting templates, so no model can reach 100%.
- **Company-name reliance** (Day 8), plus the gap between CFPB data and a single bank's tickets. Expect a drop on real bank data, and re-validate before using it there.

## 5. What would change this decision
- Render p95 above 200 ms on Nov 4 → deploy LR full-text.
- A business requirement measured in seconds, not milliseconds (asynchronous routing) → BERT is deployable even at about 1.8 s. My 200 ms target was my own assumption; confirm it with operations before the next iteration.
- Student loan's gap growing beyond 0.02 in monitoring, or fresh labelled data showing the drift has changed the ranking → re-evaluate.
